# Notebook 04 — Cleaning and Transformation

**AI Evolution Map · Data Pipeline**

---

## Purpose

Apply documented, reproducible transformations to the raw data to produce
clean intermediate records in `data/interim/` and final cleaned records
in `data/processed/`.

**Integrity rules for this notebook:**
- Raw files under `data/raw/` are NEVER modified.
- Every transformation is documented with a rationale.
- Duplicates are resolved using explicit rules, not silent drops.
- Missing values are handled according to field semantics — never filled with 0.
- Date precision is preserved: a year-only date stays as 'YYYY'.

In [1]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np
from src.cleaning.normalizers import (
    normalize_date,
    infer_date_precision,
    normalize_title,
    normalize_abstract,
    reconstruct_abstract_from_inverted_index,
    normalize_author_name,
)
from src.ingestion.openalex_client import extract_source_fields
from src.ingestion.huggingface_client import extract_model_fields

RAW_DIR = project_root / 'data' / 'raw'
INTERIM_DIR = project_root / 'data' / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print('Ready. Project root:', project_root)

Ready. Project root: C:\Users\MANAS\Documents\AI-Evolution-Map


## Transformation 1: OpenAlex Works → Interim ResearchSource Records

### Transformations applied:
1. **Column rename**: Use our field names (not OpenAlex's)
2. **Date normalization**: publication_date → YYYY-MM-DD; publication_year → YYYY fallback
3. **Date precision inference**: derive precision from the normalized date
4. **Abstract reconstruction**: from inverted index → plain text
5. **Author extraction**: flatten the nested authorships array
6. **Duplicate removal**: deduplicate by openalex_id (keep first occurrence)

### What we do NOT do:
- We do NOT infer journal references that aren't in the data
- We do NOT fill missing DOIs with guesses
- We do NOT merge different papers with similar titles

In [2]:
oa_path = RAW_DIR / 'openalex_ai_works_2018_2023.json'

if not oa_path.exists():
    print('ERROR: Run Notebook 02 first to ingest the OpenAlex data.')
else:
    with open(oa_path, encoding='utf-8') as f:
        oa_raw = json.load(f)

    # Step 1: Extract structured fields from raw API response
    extracted = [extract_source_fields(r) for r in oa_raw]
    df_oa = pd.DataFrame(extracted)
    print(f'Loaded {len(df_oa)} OpenAlex records.')

    # Step 2: Reconstruct abstract from inverted index
    # The raw record has abstract_inverted_index at the top level.
    # extract_source_fields doesn't include it, so we pull it separately.
    abstracts = [
        reconstruct_abstract_from_inverted_index(
            r.get('abstract_inverted_index')
        )
        for r in oa_raw
    ]
    df_oa['abstract'] = abstracts

    # Step 3: Normalize publication dates
    # OpenAlex provides both publication_date (YYYY-MM-DD) and publication_year (int).
    # We already have `year` from extract_source_fields. Normalize the date from raw.
    pub_dates = [
        normalize_date(r.get('publication_date'), field_name='publication_date')
        for r in oa_raw
    ]
    df_oa['publication_date_norm'] = pub_dates
    df_oa['date_precision'] = df_oa['publication_date_norm'].apply(
        lambda d: infer_date_precision(d) if d else infer_date_precision(str(df_oa.loc[0, 'year']) if df_oa.loc[0, 'year'] else None)
    )

    # Step 4: Normalize title
    df_oa['title_norm'] = df_oa['title'].apply(normalize_title)

    # Step 5: Deduplicate by openalex_id
    before_dedup = len(df_oa)
    df_oa = df_oa.drop_duplicates(subset='openalex_id', keep='first')
    removed = before_dedup - len(df_oa)
    print(f'Deduplication: removed {removed} duplicate OpenAlex IDs.')

    # Step 6: Validate our source ID was generated
    missing_ids = df_oa['our_source_id'].isnull().sum()
    print(f'Records with missing source_id: {missing_ids}')

    print(f'\nCleaned OpenAlex records: {len(df_oa)}')
    display(df_oa[['our_source_id', 'title_norm', 'year', 'publication_date_norm',
                   'date_precision', 'cited_by_count', 'venue']].head(5))

Loaded 25 OpenAlex records.
Deduplication: removed 0 duplicate OpenAlex IDs.
Records with missing source_id: 0

Cleaned OpenAlex records: 25


,our_source_id,title_norm,year,publication_date_norm,date_precision,cited_by_count,venue
0,source-openalex-w4390473182,Optimized multivariate multiscale slope entrop...,2023,2023-12-31,day,71,Chaos Solitons & Fractals
1,source-openalex-w4390467900,Detecting and diagnosing prior and likelihood ...,2023,2023-12-31,day,69,Statistics and Computing
2,source-openalex-w4390473258,Optimal power distribution control in modular ...,2023,2023-12-31,day,66,Applied Energy
3,source-openalex-w4390460610,TransVAE-DTA: Transformer and variational auto...,2023,2023-12-31,day,58,Computer Methods and Programs in Biomedicine
4,source-openalex-w4390471538,ResfEANet: ResNet-fused External Attention Net...,2023,2023-12-31,day,55,Computer Methods and Programs in Biomedicine U...


In [3]:
# Save interim OpenAlex result
if 'df_oa' in dir():
    interim_oa = INTERIM_DIR / 'openalex_works_cleaned.json'
    # Convert to records, handling any non-serializable types (lists, NaN → None)
    oa_records = df_oa.where(pd.notnull(df_oa), None).to_dict(orient='records')
    with open(interim_oa, 'w', encoding='utf-8') as f:
        json.dump(oa_records, f, indent=2, ensure_ascii=False, default=str)
    print(f'Saved {len(oa_records)} OpenAlex interim records → {interim_oa}')

Saved 25 OpenAlex interim records → C:\Users\MANAS\Documents\AI-Evolution-Map\data\interim\openalex_works_cleaned.json


## Transformation 2: arXiv Papers

In [4]:
arxiv_path = RAW_DIR / 'arxiv_cs_ai_recent_50.json'

if not arxiv_path.exists():
    print('ERROR: Run Notebook 02 first.')
else:
    with open(arxiv_path, encoding='utf-8') as f:
        arxiv_raw = json.load(f)

    df_arxiv = pd.DataFrame(arxiv_raw)

    # Normalize dates — arXiv gives YYYY-MM-DD already
    df_arxiv['published_norm'] = df_arxiv['published'].apply(
        lambda d: normalize_date(d, field_name='published')
    )
    df_arxiv['date_precision'] = df_arxiv['published_norm'].apply(infer_date_precision)

    # Normalize title
    df_arxiv['title_norm'] = df_arxiv['title'].apply(normalize_title)

    # Normalize abstract
    df_arxiv['summary_norm'] = df_arxiv['summary'].apply(normalize_abstract)

    # Normalize author names
    df_arxiv['authors_norm'] = df_arxiv['authors'].apply(
        lambda names: [normalize_author_name(n) for n in names if n] if isinstance(names, list) else []
    )

    # Deduplicate by arxiv_id
    before = len(df_arxiv)
    df_arxiv = df_arxiv.drop_duplicates(subset='arxiv_id', keep='first')
    print(f'arXiv dedup: removed {before - len(df_arxiv)} duplicates.')
    print(f'Clean arXiv records: {len(df_arxiv)}')

    interim_arxiv = INTERIM_DIR / 'arxiv_papers_cleaned.json'
    arxiv_records = df_arxiv.where(pd.notnull(df_arxiv), None).to_dict(orient='records')
    with open(interim_arxiv, 'w', encoding='utf-8') as f:
        json.dump(arxiv_records, f, indent=2, ensure_ascii=False, default=str)
    print(f'Saved → {interim_arxiv}')

    display(df_arxiv[['arxiv_id', 'title_norm', 'published_norm', 'date_precision', 'primary_category']].head(5))

arXiv dedup: removed 0 duplicates.
Clean arXiv records: 25
Saved → C:\Users\MANAS\Documents\AI-Evolution-Map\data\interim\arxiv_papers_cleaned.json


,arxiv_id,title_norm,published_norm,date_precision,primary_category
0,2610.31667,Direct Pass@$k$ Optimization for Thought Pruni...,2026-10-22,day,cs.AI
1,2610.31657,Cost-Aware Abstention for LLM-Based IT Ticket ...,2026-10-22,day,cs.LG
2,2610.31656,METTA: Memory-Enhanced Test-Time Adaptation fo...,2026-10-22,day,cs.LG
3,2610.31650,EpiSEM-Bench: Evaluating Safety and Evidence R...,2026-10-22,day,cs.AI
4,2610.31646,Low-Bit N-Gram Embeddings for Language Models,2026-10-22,day,cs.CL


## Transformation 3: Hugging Face Models

In [5]:
hf_path = RAW_DIR / 'huggingface_text_generation_top50.json'

if not hf_path.exists():
    print('ERROR: Run Notebook 02 first.')
else:
    with open(hf_path, encoding='utf-8') as f:
        hf_raw = json.load(f)

    hf_extracted = [extract_model_fields(r) for r in hf_raw]
    df_hf = pd.DataFrame(hf_extracted)

    # Normalize created_at date (HF gives ISO datetime strings)
    df_hf['created_at_norm'] = df_hf['created_at'].apply(
        lambda d: normalize_date(d, field_name='created_at')
    )
    df_hf['date_precision'] = df_hf['created_at_norm'].apply(infer_date_precision)

    # Filter out private models (should be 0 since we called public API)
    private_count = df_hf['private'].sum()
    if private_count > 0:
        print(f'Warning: {private_count} private models found — dropping.')
        df_hf = df_hf[~df_hf['private']]

    # Deduplicate by model_id
    before = len(df_hf)
    df_hf = df_hf.drop_duplicates(subset='model_id', keep='first')
    print(f'HF dedup: removed {before - len(df_hf)} duplicates. Final: {len(df_hf)} records.')

    interim_hf = INTERIM_DIR / 'huggingface_models_cleaned.json'
    hf_records_out = df_hf.where(pd.notnull(df_hf), None).to_dict(orient='records')
    with open(interim_hf, 'w', encoding='utf-8') as f:
        json.dump(hf_records_out, f, indent=2, ensure_ascii=False, default=str)
    print(f'Saved → {interim_hf}')

    display(df_hf[['model_id', 'pipeline_tag', 'created_at_norm', 'date_precision', 'downloads', 'license']].head(5))

HF dedup: removed 0 duplicates. Final: 25 records.
Saved → C:\Users\MANAS\Documents\AI-Evolution-Map\data\interim\huggingface_models_cleaned.json


,model_id,pipeline_tag,created_at_norm,date_precision,downloads,license
0,Qwen/Qwen3-0.6B,text-generation,2025-04-27,day,25158033,None
1,openai-community/gpt2,text-generation,2022-03-02,day,18836643,None
2,Qwen/Qwen3-4B,text-generation,2025-04-27,day,11396683,None
3,Qwen/Qwen3-14B,text-generation,2025-04-27,day,9401416,None
4,Qwen/Qwen3-Embedding-0.6B,feature-extraction,2025-06-03,day,8126298,None


## Transformation 4: Stanford AI Index & Epoch AI Multi-Year Historical Metrics (2013–2023)

### Transformation Rules & Methodology Integrity
1. **Strict Separation of Incompatible Metrics**: Global private AI investment is measured in constant 2021 US$ Billions (from NetBase Quid external funding tracking companies raising >$1.5M). It is never conflated with nominal figures or unadjusted series.
2. **Sector Affiliation Categorization**: Notable AI systems tracked by Epoch AI are segmented into Industry, Academia, Collaborations, and Total.
3. **Zero Interpolation**: All 11 annual observations (2013–2023) are grounded directly in official source publications. No years are fabricated or artificially smoothed.
4. **Data Contract Compliance**: Every record generates a deterministic ID, standardized metric name, annual period precision, unit, category, and source reference.


In [6]:
from src.cleaning.metrics_transformer import transform_stanford_metrics

stanford_raw_dir = RAW_DIR / 'stanford_ai_index'
if not stanford_raw_dir.exists():
    print(f'ERROR: Raw directory not found at {stanford_raw_dir}')
else:
    metrics_records, coverage_table = transform_stanford_metrics(stanford_raw_dir)
    print(f'Transformed {len(metrics_records)} multi-year metric observations across {len(coverage_table)} years.')

    interim_metrics_path = INTERIM_DIR / 'metrics_interim.json'
    with open(interim_metrics_path, 'w', encoding='utf-8') as f:
        json.dump(metrics_records, f, indent=2, ensure_ascii=False)
    print(f'Saved interim metrics → {interim_metrics_path}')

    coverage_csv_path = INTERIM_DIR / 'metrics_coverage.csv'
    coverage_table.to_csv(coverage_csv_path)
    print(f'Saved annual coverage table → {coverage_csv_path}')

    print('\n--- Annual Metric Coverage Matrix (2013-2023) ---')
    display(coverage_table)


Transformed 44 multi-year metric observations across 11 years.


Saved interim metrics → C:\Users\MANAS\Documents\AI-Evolution-Map\data\interim\metrics_interim.json
Saved annual coverage table → C:\Users\MANAS\Documents\AI-Evolution-Map\data\interim\metrics_coverage.csv

--- Annual Metric Coverage Matrix (2013-2023) ---


metric_name,global_private_ai_investment_annual_constant_2021_usd_billions,notable_ai_systems_annual_count_academia,notable_ai_systems_annual_count_industry,notable_ai_systems_annual_count_total
period,,,,
2013,1,1,1,1
2014,1,1,1,1
2015,1,1,1,1
2016,1,1,1,1
2017,1,1,1,1
2018,1,1,1,1
2019,1,1,1,1
2020,1,1,1,1
2021,1,1,1,1
